# Stress Test — 1000 Inputs × 5000 Candidates

Measures wall-clock time for matching **1000 input links** against a pool of **5000 candidate links**.

**Output**: 1000 results as `(match_id, candidate_id)` tuples, sorted by `match_id`.

In [ ]:
import random
import time

from nerds_nlp.models.edge.matching import KKEdgeMatchingModel

print("Imports OK")

## Configuration

In [ ]:
N_INPUTS = 1000
N_CANDIDATES = 5000
SEED = 42

CONFIG = {
    "matching": {
        "threshold": 0.85,
        "fields": [
            {"name": "valuation_date", "weight": 0.3, "strategy": "exact_match"},
            {
                "name": "forward_rate",
                "weight": 0.25,
                "strategy": "numeric_range",
                "params": {"tolerance": 0.01},
            },
            {"name": "settlement_date", "weight": 0.25, "strategy": "exact_match"},
            {"name": "direction", "weight": 0.2, "strategy": "direction_inverse"},
        ],
    }
}

print(f"Inputs: {N_INPUTS}, Candidates: {N_CANDIDATES}, Seed: {SEED}")

## Generate Synthetic Data

- **1000 input documents** each with 1 link
- **5000 candidate documents** each with 1 link
- The first 1000 candidates are guaranteed perfect matches (inverse direction, identical fields) for each input
- The remaining 4000 candidates are random noise

In [ ]:
random.seed(SEED)

DATES = [f"2025-{m:02d}-{d:02d}" for m in range(1, 13) for d in (1, 10, 15, 20, 28)]
RATES = [round(random.uniform(0.5, 5.0), 4) for _ in range(200)]
DIRECTION_PAIRS = [("incoming", "outgoing"), ("buy", "sell")]


def make_input_link(idx):
    pair = DIRECTION_PAIRS[idx % len(DIRECTION_PAIRS)]
    return {
        "valuation_date": DATES[idx % len(DATES)],
        "forward_rate": str(RATES[idx % len(RATES)]),
        "settlement_date": DATES[(idx + 7) % len(DATES)],
        "direction": pair[0],
    }


def make_perfect_candidate(input_link):
    """Create a candidate that is a perfect match for the input."""
    inverse_map = {"incoming": "outgoing", "outgoing": "incoming", "buy": "sell", "sell": "buy"}
    return {
        "valuation_date": input_link["valuation_date"],
        "forward_rate": input_link["forward_rate"],
        "settlement_date": input_link["settlement_date"],
        "direction": inverse_map[input_link["direction"]],
    }


def make_random_candidate():
    pair = random.choice(DIRECTION_PAIRS)
    return {
        "valuation_date": random.choice(DATES),
        "forward_rate": str(round(random.uniform(0.5, 5.0), 4)),
        "settlement_date": random.choice(DATES),
        "direction": random.choice(pair),
    }


# Build input documents
input_links = [make_input_link(i) for i in range(N_INPUTS)]
input_documents = [
    {"id": f"match-{i:04d}", "links": [input_links[i]]}
    for i in range(N_INPUTS)
]

# Build candidate documents:
# First N_INPUTS candidates are perfect matches (shuffled later in the pool)
# Remaining are random noise
candidate_documents = []
for i in range(N_INPUTS):
    candidate_documents.append(
        {"id": f"cand-{i:04d}", "links": [make_perfect_candidate(input_links[i])]}
    )
for i in range(N_INPUTS, N_CANDIDATES):
    candidate_documents.append(
        {"id": f"cand-{i:04d}", "links": [make_random_candidate()]}
    )

# Shuffle candidates so perfect matches are not at the front
random.shuffle(candidate_documents)

contract = {
    "input_documents": input_documents,
    "unmatched_documents": candidate_documents,
}

print(f"Input documents : {len(input_documents)}")
print(f"Candidate documents: {len(candidate_documents)}")
print(f"Total links to score: {N_INPUTS} x {N_CANDIDATES} = {N_INPUTS * N_CANDIDATES:,}")

## Run Matching & Measure Time

In [ ]:
model = KKEdgeMatchingModel(config_dict=CONFIG)

t_start = time.perf_counter()
results = model.match(contract)
t_end = time.perf_counter()

elapsed = t_end - t_start

print(f"Matching completed in {elapsed:.3f} seconds")
print(f"Results count   : {len(results)}")
print(f"Throughput      : {N_INPUTS / elapsed:.1f} inputs/sec")
print(f"Avg per input   : {elapsed / N_INPUTS * 1000:.2f} ms")

## Build Result Tuples — `(match_id, candidate_id)`

One tuple per input, sorted by `match_id`.

In [ ]:
# Pair each result with its input document id
result_tuples = []
idx = 0
for doc in contract["input_documents"]:
    for _ in doc["links"]:
        match_id = doc["id"]
        candidate_id = results[idx].best_candidate_document_id
        result_tuples.append((match_id, candidate_id))
        idx += 1

# Sort by match_id
result_tuples.sort(key=lambda t: t[0])

assert len(result_tuples) == N_INPUTS, (
    f"Expected {N_INPUTS} results, got {len(result_tuples)}"
)

print(f"Result length: {len(result_tuples)}")
print(f"\nFirst 10 tuples (match_id, candidate_id):")
for t in result_tuples[:10]:
    print(f"  {t}")
print(f"\nLast 10 tuples (match_id, candidate_id):")
for t in result_tuples[-10:]:
    print(f"  {t}")

## Validate Results

In [ ]:
# Verify length
assert len(result_tuples) == 1000, f"Expected 1000 results, got {len(result_tuples)}"

# Verify sorted by match_id
match_ids = [t[0] for t in result_tuples]
assert match_ids == sorted(match_ids), "Results are not sorted by match_id"

# Verify each perfect-match input found its intended candidate
matched_count = sum(1 for r in results if r.matched)
unmatched_count = sum(1 for r in results if not r.matched)

print(f"Matched   : {matched_count}")
print(f"Unmatched : {unmatched_count}")

# Check that perfect-match candidates were found
# Input match-XXXX should match cand-XXXX
correct = 0
for match_id, candidate_id in result_tuples:
    input_num = match_id.split("-")[1]
    if candidate_id is not None and candidate_id == f"cand-{input_num}":
        correct += 1

print(f"\nCorrect matches (input N -> cand N): {correct} / {N_INPUTS}")
print(f"Accuracy: {correct / N_INPUTS * 100:.1f}%")

# Score distribution
scores = [r.best_score for r in results]
print(f"\nScore distribution:")
print(f"  Min   : {min(scores):.4f}")
print(f"  Max   : {max(scores):.4f}")
print(f"  Mean  : {sum(scores) / len(scores):.4f}")

print(f"\nStress test completed in {elapsed:.3f}s")
print("All assertions passed.")